In [3]:
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

import pandas as pd
from pathlib import Path

In [4]:
data_dir = Path("../data")
silver_dir = data_dir / "silver"
training_data = pd.read_csv(silver_dir / "train_data.csv")
test_data = pd.read_csv(silver_dir / "test_data.csv")

training_data["attack_type"] = training_data["attack_type"].apply(
    lambda x: "normal." if x == "normal." else "attack."
)

test_data["attack_type"] = test_data["attack_type"].apply(
    lambda x: "normal." if x == "normal." else "attack."
)

for col in ['protocol_type', 'service', 'flag']:
    le = LabelEncoder()
    training_data[col] = le.fit_transform(training_data[col])
    test_data[col] = le.fit_transform(test_data[col])

X_train = training_data.drop(columns=["attack_type"])
y_train = training_data["attack_type"]

X_test = test_data.drop(columns=["attack_type"])
y_test = test_data["attack_type"]

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack_type
0,0,1,22,9,181,5450,0,0,0,0,...,9,1.0,0.0,0.11,0.0,0.0,0.0,0.0,0.0,normal.
1,0,1,22,9,239,486,0,0,0,0,...,19,1.0,0.0,0.05,0.0,0.0,0.0,0.0,0.0,normal.
2,0,1,22,9,235,1337,0,0,0,0,...,29,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,normal.
3,0,1,22,9,219,1337,0,0,0,0,...,39,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,normal.
4,0,1,22,9,217,2032,0,0,0,0,...,49,1.0,0.0,0.02,0.0,0.0,0.0,0.0,0.0,normal.


In [6]:
param_grid = {
    "hidden_layer_sizes": [(50,), (100,), (100, 50)],
    "activation": ["relu", "tanh"],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate_init": [0.001, 0.01],
}
mlp_classifier = MLPClassifier()

mlp_models = GridSearchCV(
    mlp_classifier,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

mlp_models.fit(X_train, y_train)
print("Best Parameters:", mlp_models.best_params_)

best_model = mlp_models.best_estimator_

prediction = best_model.predict(X_test)
print(classification_report(y_test, prediction))

In [7]:
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [5, 10, 20, 30],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10]
}
dt_classifier = DecisionTreeClassifier(random_state=42)

dt_models = GridSearchCV(
    dt_classifier,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

dt_models.fit(X_train, y_train)
print("Best Parameters:", dt_models.best_params_)

best_model = dt_models.best_estimator_

prediction = best_model.predict(X_test)
print(classification_report(y_test, prediction))

C:\Users\ryuka\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_search.py:1234: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan]
  warnings.warn(


Best Parameters: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}
              precision    recall  f1-score   support

     attack.       1.00      0.91      0.95    250436
     normal.       0.73      0.99      0.84     60593

    accuracy                           0.93    311029
   macro avg       0.86      0.95      0.90    311029
weighted avg       0.95      0.93      0.93    311029



In [8]:
param_grid = {
    "n_estimators": [100, 200, 500],
    "max_depth": [5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

rf_classifier = RandomForestClassifier(random_state=42)

rf_models = GridSearchCV(
    rf_classifier,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

rf_models.fit(X_train, y_train)
print("Best Parameters:", rf_models.best_params_)

best_model = rf_models.best_estimator_
prediction = best_model.predict(X_test)
print(classification_report(y_test, prediction))

In [9]:
param_grid = {
    "n_estimators": [100, 200, 500],
    "max_depth": [5, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

et_classifier = ExtraTreesClassifier(random_state=42)

et_models = GridSearchCV(
    et_classifier,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

et_models.fit(X_train, y_train)
print("Best Parameters:", et_models.best_params_)

best_model = et_models.best_estimator_
prediction = best_model.predict(X_test)
print(classification_report(y_test, prediction))

In [10]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [3, 5],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5]
}

gb_classifier = GradientBoostingClassifier(random_state=42)

gb_models = GridSearchCV(
    gb_classifier,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

gb_models.fit(X_train, y_train)
print("Best Parameters:", gb_models.best_params_)

best_model = gb_models.best_estimator_
prediction = best_model.predict(X_test)
print(classification_report(y_test, prediction))